#  LLM 성능평가 개요

### **학습 목표:** A/B 테스트를 수행하여 LLM 애플리케이션의 성능 평가를 적용한다

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

`(3) langfuase handler 설정`

In [ ]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

`(4) Test Data`

In [ ]:
# Test 데이터셋에 대한 QA 생성 결과를 리뷰한 후 다시 로드
import pandas as pd
df_qa_test = pd.read_excel("data/testset.xlsx")

print(f"테스트셋: {df_qa_test.shape[0]}개 문서")
df_qa_test.head(2)

---

## **LLM 애플리케이션 평가**

- **AI 평가**는 데이터셋, 평가자, 평가 방법론 세 가지 핵심 요소로 구성되며, 초기에는 **10-20개의 고품질 예제**로 시작하는 것이 효과적

- 평가 방식은 **인간 평가**와 **자동화 평가** 두 트랙으로 진행되며, 주관적 판단이 필요한 초기에는 인간 평가를, 확장이 필요한 경우 휴리스틱 기반 자동화 평가를 활용

- 평가는 **오프라인**과 **온라인** 환경에서 수행되며, 벤치마킹, 테스트, 실시간 모니터링 등 상황에 맞는 방법론을 적용

- 지속적인 **CI/CD 통합**과 모니터링 시스템 구축을 통해 평가 프로세스의 효율성과 신뢰성을 확보해야 함

### 1) **벡터스토어** 로드

- **Chroma DB** 설정에서 모델, 컬렉션명, 저장 경로 지정

In [ ]:
# 벡터 저장소 로드
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

In [ ]:
# 벡터저장소 검색기 생성
chroma_k = chroma_db.as_retriever(
    search_kwargs={'k': 4},
)

# 벡터저장소 검색기를 사용하여 검색
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"

retrieved_docs = chroma_k.invoke(query)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")
    print("-"*200)
    print()

### 2) **BM25 검색기** 준비

- **BM25 검색기** 구현으로 문서 유사도 기반 검색 가능

- **한국어 텍스트 처리**를 위한 **Kiwi 토크나이저** 설정

- 참고: https://github.com/bab2min/kiwipiepy

In [ ]:
# korean_docs 파일을 로드 (jsonlines 파일)
def load_jsonlines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        docs = [json.loads(line) for line in f]
    return docs

korean_docs = load_jsonlines('data/korean_docs_final.jsonl')
print(f"로드된 문서: {len(korean_docs)}개")
pprint(korean_docs[0])

In [ ]:
from langchain.schema import Document  # Document 클래스 임포트

# 문자열 리스트를 Document 객체로 변환
if isinstance(korean_docs[0], str):  # 첫 번째 항목이 문자열인지 확인
    documents = [
        Document(
            page_content=json.loads(data)['page_content'],  # 문자열을 파이썬 객체로 변환
            metadata=json.loads(data)['metadata']
        )
        for i, data in enumerate(korean_docs)
    ]
else:
    documents = korean_docs

print(f"변환된 문서: {len(documents)}개")
pprint(documents[0])

In [ ]:
# BM25 검색기를 사용하기 위한 준비
from krag.tokenizers import KiwiTokenizer
from krag.retrievers import KiWiBM25RetrieverWithScore

kiwi_tokenizer = KiwiTokenizer(
    model_type='knlm',    # Kiwi 언어 모델 타입
    typos='basic'         # 기본 오타교정
    )

bm25_db = KiWiBM25RetrieverWithScore(
        documents=documents,
        kiwi_tokenizer=kiwi_tokenizer,
        k=4,
    )

In [ ]:
# BM25 검색기를 사용하여 문서 검색
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
retrieved_docs = bm25_db.invoke(query, 2)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"BM25 점수: {doc.metadata["bm25_score"]:.2f}")
    print(f"\n{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*200)

### 3) **Emsemble Hybrid Search** 준비

- **BM25**, **벡터 검색** 결과를 **rank-fusion** 알고리즘으로 통합 (**EnsembleRetriever**)

- 각 검색기의 **순위 점수**를 고려한 최종 순위 결정

- **중복 문서** 제거와 **재순위화** 자동 수행

- 두 검색 방식의 **장점을 결합**해 검색 품질 향상

In [ ]:
from langchain.retrievers import EnsembleRetriever

# 검색기 초기화
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_db, chroma_k],
    weights=[0.5, 0.5],
)

In [ ]:
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
retrieved_docs = hybrid_retriever.invoke(query)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"\n{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*200)

### 4) **RAG 체인**

- **답변**과 **검색 문서**를 함께 출력

In [ ]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import RunnableConfig, RunnablePassthrough, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import List, Dict

def rag_bot(
    question: str,
    retriever: BaseRetriever,
    llm: BaseChatModel,
    config: RunnableConfig | None = None,
) -> Dict[str, str | List[Document]]:
    """
    문서 검색 기반 질의응답 수행
    """
    docs = retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)

    system_prompt = f"""문서 기반 질의응답 어시스턴트입니다.
- 제공된 문서만 참고하여 답변
- 불확실할 경우 '모르겠습니다' 라고 응답
- 3문장 이내로 답변

[문서]
{context}"""

    prompt = ChatPromptTemplate.from_messages(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "\n\n[질문]{question}\n\n[답변]\n"},
        ]
    )

    docqa_chain = {
        "context": lambda x: context,
        "question": RunnablePassthrough(),
        "docs": lambda x: docs,
    } | RunnableParallel({
        "answer": prompt | llm | StrOutputParser(),
        "documents": lambda x: x["docs"],
    })

    return docqa_chain.invoke(question, config=config)

In [ ]:
from langchain_openai import ChatOpenAI

# 모델 생성
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# RAG 체인 실행
rag_bot(
    question="Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?",
    retriever=hybrid_retriever,
    llm=llm,
    config={"callbacks": [langfuse_handler]},   # 콜백 핸들러 추가
)

---
### **[실습]**

- gemin-1.5-flash 모델과 벡터스토어 검색기를 사용하여 RAG 체인을 실행합니다.
- 실행 결과를 langfuse UI에서 확인하고, gpt-4.1-mini 모델의 답변과 비교합니다.

In [ ]:
# 여기에 코드를 작성하세요.
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import RunnableConfig, RunnablePassthrough, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import List, Dict

def rag_bot(
    question: str,
    retriever: BaseRetriever,
    llm: BaseChatModel,
    config: RunnableConfig | None = None,
) -> Dict[str, str | List[Document]]:
    """
    문서 검색 기반 질의응답 수행
    """
    docs = retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)

    system_prompt = f"""문서 기반 질의응답 어시스턴트입니다.
- 제공된 문서만 참고하여 답변
- 불확실할 경우 '모르겠습니다' 라고 응답
- 3문장 이내로 답변

[문서]
{context}"""

    prompt = ChatPromptTemplate.from_messages(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "\n\n[질문]{question}\n\n[답변]\n"},
        ]
    )

    docqa_chain = {
        "context": lambda x: context,
        "question": RunnablePassthrough(),
        "docs": lambda x: docs,
    } | RunnableParallel({
        "answer": prompt | llm | StrOutputParser(),
        "documents": lambda x: x["docs"],
    })

    return docqa_chain.invoke(question, config=config)

from langchain_google_genai import ChatGoogleGenerativeAI

# 모델 생성
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0)

# RAG 체인 실행
rag_bot(
    question="Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?",
    retriever=hybrid_retriever,
    llm=llm,
    config={"callbacks": [langfuse_handler]},   # 콜백 핸들러 추가
)

---

## **Comparison (비교 평가)**

- LangChain **비교 평가기**로 동일 입력에 대한 여러 모델의 출력을 객관적 비교

- **성능 차이** 분석을 통해 최적의 모델과 프롬프트 선택

- **A/B 테스트**와 **선호도 점수** 생성에 활용되며 LangSmith와 통합

- 평가 결과(형식):
    - value: 선호되는 응답 ('A' 또는 'B'), 선호되는 응답이 없는 경우 ('C')
    - score: 0 또는 1 (1은 첫 번째 예측이 선호됨을 의미)
    - reasoning: 평가 근거에 대한 상세한 설명

- 참조: https://python.langchain.com/api_reference/langchain/evaluation.html

### **1) Reference-free**

- **평가 특징**: 참조 답변 없이 두 RAG 답변 직접 비교
- **평가 요소**: 사실성, 관련성, 일관성 등 상대 비교
- **장점**: 절대 기준 없이도 RAG 시스템 간 성능 차이 판단 가능 (객관적 비교 가능)

`(1) A/B 테스트 평가 - 기본 개요`

In [ ]:
from langchain.evaluation import load_evaluator
from langchain_google_genai import ChatGoogleGenerativeAI

# 비교 평가기 생성
evaluator = load_evaluator(
    "pairwise_string",
    llm=ChatGoogleGenerativeAI(model="gemini-1.5-pro"),
    callbacks=[langfuse_handler],
)

# 두 모델의 출력 비교
result = evaluator.evaluate_string_pairs(
    prediction="파이썬은 읽기 쉽고 간단한 문법을 가진 프로그래밍 언어입니다.",
    prediction_b="파이썬은 동적 타이핑을 지원하는 고수준 프로그래밍 언어로, 데이터 과학과 웹 개발에 널리 사용됩니다.",
    input="파이썬이 무엇인지 설명해주세요."
)

print(f"선호되는 응답: {result['value']} ({result['score']})")
print("-"*200)
print(f"평가 근거: {result['reasoning']}")
print("="*200)

`(2) A/B 테스트 평가 - 모델 비교`

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM 모델 생성
gpt4omini_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
gemini15flash_llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

# RAG 체인 실행
question = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
gpt_response = rag_bot(
    question=question,
    retriever=hybrid_retriever,
    llm=gpt4omini_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation"],
        "metadata": {
            "model": "gpt-4.1-mini",
            "temperature": 0,
            },
        },
)


gemini_response = rag_bot(
    question=question,
    retriever=hybrid_retriever,
    llm=gemini15flash_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation"],
        "metadata": {
            "model": "gemini-1.5-flash",
            "temperature": 0,
            },
        },
)

# 비교 평가기 생성 : OpenAI gpt-4.1-mini vs Google Gemini-1.5-flash
evaluator = load_evaluator(
    "pairwise_string",
    llm=ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0),  # Google Gemini-1.5-pro 를 평가자로 사용
    callbacks=[langfuse_handler],
    tags=["rag_bot", "evaluation", "pairwise_string"],
    metadata={
        "model_a": "gpt-4.1-mini",
        "model_b": "gemini-1.5-flash",
        "evaluator": "gemini-1.5-pro",
    },
)

# 두 모델의 출력 비교
result = evaluator.evaluate_string_pairs(
    prediction=gpt_response["answer"],
    prediction_b=gemini_response["answer"],
    input=question
)

print(f"gpt-4.1-mini: {gpt_response['answer']}")
print(f"Gemini-1.5-flash: {gemini_response['answer']}")
print("-"*200)
print(f"선호되는 응답: {result['value']} ({result['score']})")
print(f"평가 근거: {result['reasoning']}")
print("="*200)

---
### **[실습]**

- ollama와 groq에서 각각 1개의 모델을 선택하고, RAG 체인을 구성합니다.
- 두 모델의 실행 결과를 비교합니다. (langfuse UI 확인)

In [ ]:
# 여기에 코드를 작성하세요.

`(3) A/B 테스트 평가 - 프롬프트 비교`

In [ ]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import RunnableConfig
from typing import List, Dict

def rag_bot_b(
    question: str,
    retriever: BaseRetriever,
    llm: BaseChatModel,
    config: RunnableConfig | None = None,
) -> Dict[str, str | List[Document]]:
    """
    문서 검색 기반 질의응답 수행
    """
    docs = retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)

    system_prompt = f"""RAG Assistant to answer questions based on provided documents.

    Guidelines:
    - Reference only provided context
    - Reply "I don't know" if uncertain
    - Keep responses under 3 sentences

    Context:
    {context}"""

    prompt = ChatPromptTemplate.from_messages(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "\n\n[Question]{question}\n\n[Answer]\n"},
        ]
    )

    docqa_chain = {
        "context": lambda x: context,
        "question": RunnablePassthrough(),
        "docs": lambda x: docs,
    } | RunnableParallel({
        "answer": prompt | llm | StrOutputParser(),
        "documents": lambda x: x["docs"],
    })

    return docqa_chain.invoke(question, config=config)

In [ ]:
# RAG 체인 실행
question = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
gpt_prompt_b_response = rag_bot_b(
    question=question,
    retriever=hybrid_retriever,
    llm=gpt4omini_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation", "prompt_b"],
        "metadata": {
            "model": "gpt-4.1-mini",
            "temperature": 0,
        },
    },
)

# 두 모델의 출력 비교 (프롬프트 성능 비교)
result = evaluator.evaluate_string_pairs(
    prediction=gpt_response["answer"],
    prediction_b=gpt_prompt_b_response["answer"],
    input=question
)

print(f"gpt-4.1-mini: {gpt_response['answer']}")
print(f"gpt-4.1-mini (Prompt B): {gpt_prompt_b_response['answer']}")
print("-"*200)
print(f"선호되는 응답: {result['value']} ({result['score']})")
print(f"평가 근거: {result['reasoning']}")
print("="*200)

---
### **[실습]**

- 두 가지 버전의 프롬프트를 작성하고, 각각 별도의 RAG 체인을 구성합니다. (모델은 공통 적용)
- 두 가지 실행 결과를 비교합니다. (langfuse UI 확인)

In [ ]:
# 여기에 코드를 작성하세요.

### **2) Reference-based**

- **기준 활용**: 참조 답안과 RAG 응답을 비교 평가
- **평가 방식**: 자동화된 A/B 테스트로 객관적 성능 측정
- **주요 지표**: 정확도, 완성도, 관련성 등 정량적 평가
- 참조 답안 기반 **체계적인 품질 평가** 수행

`(1) A/B 테스트 평가 - 모델 비교`

In [ ]:
# 기본 A/B 테스트 평가 예제 (참조 답안 사용: reference-based scoring)
from langchain.evaluation import load_evaluator
from langchain_google_genai import ChatGoogleGenerativeAI

# 참조 답안이 있는 평가기 생성
evaluator = load_evaluator(
    "labeled_pairwise_string",
    llm=ChatGoogleGenerativeAI(model="gemini-1.5-flash")
)

# 참조 답안
ground_truth = """Elon Musk는 2004년 2월에 750만 달러의 시리즈 A 자금 조달을 주도하여 Tesla의 회장 겸 최대 주주가 되었습니다.
그는 주류 차량으로 확장하기 전에 프리미엄 스포츠카로 시작하는 전략에 초점을 맞춰 적극적인 역할을 수행했습니다. 2008년 10월에는 CEO로 인수했습니다.
그러나 Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다."""

# 두 모델의 출력 비교
result = evaluator.evaluate_string_pairs(
    prediction=gpt_response["answer"],        # gpt-4.1-mini 응답 (A)
    prediction_b=gemini_response["answer"],   # Gemini-1.5-flash 응답 (B)
    input=question,       # 질문
    reference=ground_truth  # 참조 답안 (정답)
)

print(f"gpt-4.1-mini: {gpt_response['answer']}")
print(f"Gemini-1.5-flash: {gemini_response['answer']}")
print("-"*200)
print(f"선호되는 응답: {result['value']} ({result['score']})")
print(f"평가 근거: {result['reasoning']}")
print("="*200)

### **[실습]**

- 테스트셋(df_qa_test)에서 하나의 샘플을 선택합니다.
- 이 샘플에 대한 RAG 답변을 두 가지 모델로부터 구합니다.
- 두 가지 실행결과에 대한 A/B 테스트 분석을 수행합니다. (langfuse UI 확인)

In [ ]:
# 여기에 코드를 작성하세요.

`(2) 사용자 정의 기준으로 A/B 테스트 평가 - 모델 비교`

In [ ]:
# 사용자 정의 평가 기준 정의
custom_criteria = {
    "간결성": "문장이 간단하고 불필요한 내용이 없는가?",
    "명확성": "문장이 명확하고 이해하기 쉬운가?",
    "정확성": "내용이 정확하고 사실에 부합하는가?",
    "적절성": "글의 어조와 스타일이 적절한가?",
}

# 사용자 정의 평가 기준을 사용하여 평가기 생성
evaluator = load_evaluator(
    "labeled_pairwise_string",
    criteria=custom_criteria,
    llm=ChatGoogleGenerativeAI(model="gemini-1.5-pro")
)

# 두 모델의 출력 비교
result = evaluator.evaluate_string_pairs(
    prediction=gpt_response["answer"],        # gpt-4.1-mini 응답 (A)
    prediction_b=gemini_response["answer"],   # Gemini-1.5-flash 응답 (B)
    input=question,       # 질문
    reference=ground_truth  # 참조 답안 (정답)
)

print(f"gpt-4.1-mini: {gpt_response['answer']}")
print(f"Gemini-1.5-flash: {gemini_response['answer']}")
print("-"*200)
print(f"선호되는 응답: {result['value']} ({result['score']})")
print(f"평가 근거: {result['reasoning']}")
print("="*200)

---
### **[실습]**

- 테스트셋(df_qa_test)에서 하나의 샘플을 선택합니다.
- 이 샘플에 대한 RAG 답변을 두 가지 모델로부터 구합니다.
- 두 가지 실행결과에 대한 A/B 테스트 분석을 수행합니다. (langfuse UI 확인)

In [ ]:
# 여기에 코드를 작성하세요.

`(3) 사용자 정의 프롬프트로 A/B 테스트 평가 - 모델 비교`

In [ ]:
from langchain_core.prompts import PromptTemplate

# 사용자 정의 프롬프트 템플릿 생성
prompt_template = PromptTemplate.from_template(
    """주어진 입력 맥락에서 A와 B 중 어느 것이 더 나은지 평가하시오.

다음 기준에 따라 평가하시오:
{criteria}

단계별로 평가한 후, 마지막 줄에 [[A]] 또는 [[B]]로 응답하시오.

데이터
----
입력: {input}
참조: {reference}
A: {prediction}
B: {prediction_b}
----
평가 근거:
"""
)

# 사용자 정의 평가 기준 정의
custom_criteria = {
    "간결성": "문장이 간단하고 불필요한 내용이 없는가?",
    "명확성": "문장이 명확하고 이해하기 쉬운가?",
    "정확성": "내용이 정확하고 사실에 부합하는가?",
    "적절성": "글의 어조와 스타일이 적절한가?",
}

# 평가기 생성
evaluator = load_evaluator(
    "labeled_pairwise_string",
    criteria=custom_criteria,
    prompt=prompt_template,
    llm=ChatGoogleGenerativeAI(model="gemini-1.5-flash"),
)

# 두 모델의 출력 비교
result = evaluator.evaluate_string_pairs(
    prediction=gpt_response["answer"],        # gpt-4.1-mini 응답 (A)
    prediction_b=gemini_response["answer"],   # Gemini-1.5-flash 응답 (B)
    input=question,       # 질문
    reference=ground_truth  # 참조 답안 (정답)
)

print(f"gpt-4.1-mini: {gpt_response['answer']}")
print(f"Gemini-1.5-flash: {gemini_response['answer']}")
print("-"*200)
print(f"선호되는 응답: {result['value']} ({result['score']})")
print(f"평가 근거: {result['reasoning']}")
print("="*200)

### **[실습]**

- 테스트셋(df_qa_test)에서 하나의 샘플을 선택합니다.
- 이 샘플에 대한 RAG 답변을 두 가지 모델로부터 구합니다.
- 두 가지 실행결과에 대한 A/B 테스트 분석을 수행합니다. (langfuse UI 확인)

In [ ]:
# 여기에 코드를 작성하세요.

---

## [실습] **RAG 성능 A/B 테스트**

- **LangChain 평가기**를 사용하여 RAG 답변의 품질을 평가합니다.

- 다음과 같은 **사용자 정의 평가 기준**을 정의하여 평가합니다. (예시)
    - Conciseness (간결성): 불필요한 반복이나 장황함 없이 핵심 내용 전달
    - Helpfulness (유용성): 실질적인 도움이 되는 정도
    - Harmfulness/Maliciousness (유해성): 해로운 내용 포함 여부

- 요구 사항:
    - 올라마(Ollama)에서 다운로드한 오픈소스 모델 성능을 gpt-4.1-mini 모델의 성능과 비교
    - 평가자 모델은 gpt-4.1 사용
    - 사용자 정의 프롬프트 사용
    - df_qa_test 전체 테스트셋에 대해서 평가를 수행
    - Reference-free 평가와 Reference-based 평가를 각각 수행 (1개 이상)


In [ ]:
# 여기에 코드를 작성하세요.